# Day 30: Fine‑tuning with RLHF – Reward Model Training (Conceptual)

Note: Full training requires significant compute. This notebook demonstrates the pipeline with a small subset for illustration.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
from trl import RewardTrainer, RewardConfig
import os

## 1. Load a preference dataset (Anthropic HH‑RLHF, small sample)

In [ ]:
# Load a small subset for demo
dataset = load_dataset("Anthropic/hh-rlhf", split="train")
dataset = dataset.select(range(100))  # use only 100 examples for speed
print(dataset[0].keys())

## 2. Preprocess for reward modelling
Reward model expects `chosen` and `rejected` responses given the same prompt.

In [ ]:
def preprocess_function(examples):
    # The dataset has 'chosen' and 'rejected' fields (full conversations)
    return {
        "input_ids_chosen": tokenizer(examples["chosen"], truncation=True, padding="max_length", max_length=512)["input_ids"],
        "input_ids_rejected": tokenizer(examples["rejected"], truncation=True, padding="max_length", max_length=512)["input_ids"],
    }

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

dataset = dataset.map(preprocess_function, batched=True)
dataset.set_format(type="torch", columns=["input_ids_chosen", "input_ids_rejected"])

## 3. Load a base model as reward model
We add a classification head with one output (reward score).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=1,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)
model.config.pad_token_id = tokenizer.eos_token_id

## 4. Configure and run RewardTrainer

In [ ]:
reward_config = RewardConfig(
    output_dir="./reward_model",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=10,
    remove_unused_columns=False,
)

trainer = RewardTrainer(
    model=model,
    tokenizer=tokenizer,
    args=reward_config,
    train_dataset=dataset,
)

# Uncomment to actually train (takes time/memory):
# trainer.train()
# trainer.save_model("./reward_model_final")
print("Training ready – uncomment to run.")

## 5. Using the trained reward model to score responses

In [ ]:
def score_response(response_text):
    inputs = tokenizer(response_text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        score = model(**inputs).logits[0].item()
    return score

# Example (model not yet trained, scores are random)
print("Score for 'Hello':", score_response("Hello"))
print("Score for 'I hate you':", score_response("I hate you"))